# GloGEM vs GloGEMflow: Aletsch & Morteratsch

Compares two geometry-evolution approaches on the two Alpine test glaciers:

| Model | Geometry approach |
|-------|-------------------|
| **GloGEM Δh** | Δh parameterisation (Huss et al. 2010): distributes volume change across elevation bands via a normalised elevation-range function; no explicit flow dynamics |
| **GloGEMflow** | SIA flowline dynamics: full ice-flow calibration with A_flow + ELA-bias spinup (Zekollari et al. 2019) |

Both models use identical calibrated MB parameters (DDFs, C_prec, T_offset) from the Hugonnet 2000–2020 calibration.

**Sections**
1. Full mass balance timeseries (1940–2100)
2. Annual MB comparison over the Hugonnet period (2000–2019)
3. Geometry evolution: area, volume, ELA
4. Mean surface elevation change rate (dh/dt) over the Hugonnet period
5. 2100 projections

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import rcParams
from matplotlib.ticker import MaxNLocator

try:
    from cmcrameri import cm as cmc
    COL_DH   = cmc.roma(0.92)   # deep blue
    COL_FLOW  = cmc.roma(0.08)  # warm orange-brown
except ImportError:
    print('cmcrameri not found — using fallback colours')
    COL_DH   = '#1565C0'
    COL_FLOW  = '#B71C1C'

COL_HUG    = '0.35'   # dark grey for Hugonnet reference
ALPHA_BAND  = 0.15    # uncertainty band fill
ALPHA_SHADE = 0.18    # Hugonnet period background shade

# Arial is not installed on this system; Liberation Sans is metrically identical
rcParams['font.family']       = 'sans-serif'
rcParams['font.sans-serif']   = ['Liberation Sans', 'Arial', 'DejaVu Sans']
rcParams['axes.unicode_minus'] = False
rcParams['figure.dpi']        = 200
rcParams['axes.spines.top']   = False
rcParams['axes.spines.right'] = False
rcParams['axes.labelsize']    = 11
rcParams['legend.framealpha'] = 0.9
rcParams['legend.fontsize']   = 9
rcParams['axes.grid']         = True
rcParams['grid.alpha']        = 0.35
rcParams['grid.linestyle']    = '--'
rcParams['grid.linewidth']    = 0.5

In [ ]:
# ── Paths ─────────────────────────────────────────────────────────────────
_base = '/scratch_net/vierzack04_fourth/jabeer/GloGEM/glogemflow_development'
_sub  = 'monthly/CentralEurope/files/files_original/BCC-CSM2-MR/ssp126'
DIR_DH   = f'{_base}/dhdt/{_sub}/'
DIR_FLOW  = f'{_base}/flow/{_sub}/'
DIR_HUG   = '/home/jabeer/projects/glogemflow_development/GloGEM/test/data/geodetic/RGIv7.0/aggregated_2000_2020/'

# ── File names ─────────────────────────────────────────────────────────────
RUN_SUFFIX = '_r1_Aletsch_Morteratsch'

# ── Time axis ──────────────────────────────────────────────────────────────
TRAN0   = 1940
N_YEARS = 161    # 1940–2100 inclusive
years   = np.arange(TRAN0, TRAN0 + N_YEARS)

# ── Glaciers ───────────────────────────────────────────────────────────────
GLACIER_IDS   = ['02596', '02216']
GLACIER_NAMES = ['Aletsch', 'Morteratsch']

# ── Hugonnet period ────────────────────────────────────────────────────────
HUG_PERIOD = (2000, 2019)
i_hug0     = HUG_PERIOD[0] - TRAN0
i_hug1     = HUG_PERIOD[1] - TRAN0 + 1   # exclusive slice end
years_hug  = years[i_hug0:i_hug1]

print('Configuration ready.')
print(f'  Δh param. dir: {DIR_DH}')
print(f'  flow dir:      {DIR_FLOW}')

In [3]:
def read_glogem_row(filepath, glacier_id):
    """Return annual timeseries for one glacier from a GloGEM .dat file."""
    with open(filepath) as f:
        f.readline()  # skip header line
        for line in f:
            parts = line.split()
            if parts and parts[0] == str(glacier_id):
                return np.array(parts[1:], dtype=float)
    raise ValueError(f'Glacier {glacier_id} not found in {filepath}')


def read_hugonnet_mb(directory, glacier_id):
    """Return (MB, err) in m w.e. yr-1 from Hugonnet 2000-2020 file."""
    filepath = directory + '11_mb_glspec.dat'
    with open(filepath) as f:
        for _ in range(3):
            f.readline()  # skip 3 header lines
        for line in f:
            parts = line.split()
            if len(parts) < 6:
                continue
            gid = parts[0][9:14]  # RGI70-11.XXXXX → 5-char suffix
            if gid == str(glacier_id):
                return float(parts[4]), float(parts[5])
    raise ValueError(f'Glacier {glacier_id} not found in Hugonnet file')


print('Helper functions defined.')

Helper functions defined.


In [ ]:
# Load all timeseries — shape (n_glaciers, n_years)
mb_dh     = np.array([read_glogem_row(DIR_DH + f'centraleurope_Annual_Balance_sfc{RUN_SUFFIX}.dat', g) for g in GLACIER_IDS])
mb_flow   = np.array([read_glogem_row(DIR_FLOW  + f'centraleurope_Annual_Balance_sfc{RUN_SUFFIX}.dat', g) for g in GLACIER_IDS])
area_dh   = np.array([read_glogem_row(DIR_DH + f'centraleurope_Area{RUN_SUFFIX}.dat', g)             for g in GLACIER_IDS])
area_flow = np.array([read_glogem_row(DIR_FLOW  + f'centraleurope_Area{RUN_SUFFIX}.dat', g)             for g in GLACIER_IDS])
vol_dh    = np.array([read_glogem_row(DIR_DH + f'centraleurope_Volume{RUN_SUFFIX}.dat', g)           for g in GLACIER_IDS])
vol_flow  = np.array([read_glogem_row(DIR_FLOW  + f'centraleurope_Volume{RUN_SUFFIX}.dat', g)           for g in GLACIER_IDS])
ela_dh    = np.array([read_glogem_row(DIR_DH + f'centraleurope_ELA{RUN_SUFFIX}.dat', g)              for g in GLACIER_IDS])
ela_flow  = np.array([read_glogem_row(DIR_FLOW  + f'centraleurope_ELA{RUN_SUFFIX}.dat', g)              for g in GLACIER_IDS])

hug_mb, hug_err = zip(*[read_hugonnet_mb(DIR_HUG, g) for g in GLACIER_IDS])
hug_mb  = np.array(hug_mb)
hug_err = np.array(hug_err)

# ── Bias summary table ─────────────────────────────────────────────────────
print(f"{'Glacier':<14} {'Hugonnet':>10} {'GloGEM Δh':>11} {'GloGEMflow':>12} {'bias Δh':>9} {'bias flow':>11}")
print('-' * 74)
for g, name in enumerate(GLACIER_NAMES):
    m_d = mb_dh[g,   i_hug0:i_hug1].mean()
    m_f = mb_flow[g, i_hug0:i_hug1].mean()
    print(f'{name:<14} {hug_mb[g]:>10.3f} {m_d:>11.3f} {m_f:>12.3f} {m_d-hug_mb[g]:>+9.3f} {m_f-hug_mb[g]:>+11.3f}')
print('\nAll values in m w.e. yr-1')

## 1. Full mass balance timeseries (1940–2100)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5), sharex=True, sharey=False)

for g, (ax, name) in enumerate(zip(axes, GLACIER_NAMES)):
    # Hugonnet period background
    ax.axvspan(HUG_PERIOD[0], HUG_PERIOD[1], color='0.85', alpha=ALPHA_SHADE, zorder=0)
    # Hugonnet reference
    ax.axhspan(hug_mb[g] - hug_err[g], hug_mb[g] + hug_err[g],
               color=COL_HUG, alpha=ALPHA_BAND, zorder=1)
    ax.axhline(hug_mb[g], color=COL_HUG, lw=1.5, ls='--', zorder=2)
    # Model lines
    ax.plot(years, mb_dh[g],   color=COL_DH,   lw=1.5, label='GloGEM Δh param.', zorder=3)
    ax.plot(years, mb_flow[g], color=COL_FLOW,  lw=1.5, label='GloGEMflow',       zorder=3)
    ax.set_title(name, fontsize=12, fontweight='bold')
    ax.set_xlabel('Year')
    ax.xaxis.set_major_locator(MaxNLocator(integer=True))
    if g == 0:
        ax.set_ylabel('Annual MB (m w.e. yr$^{-1}$)')
    ax.legend(loc='upper right', fontsize=9)

fig.tight_layout()
plt.show()

## 2. Annual MB over the Hugonnet period (2000–2019)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5), sharex=True, sharey=False)

for g, (ax, name) in enumerate(zip(axes, GLACIER_NAMES)):
    mb_d = mb_dh[g,   i_hug0:i_hug1]
    mb_f = mb_flow[g, i_hug0:i_hug1]
    # Hugonnet reference band
    ax.axhspan(hug_mb[g] - hug_err[g], hug_mb[g] + hug_err[g],
               color=COL_HUG, alpha=ALPHA_BAND, zorder=1, label='Hugonnet ±1σ')
    ax.axhline(hug_mb[g], color=COL_HUG, lw=1.5, ls='--', zorder=2, label='Hugonnet mean')
    # Model lines with markers
    ax.plot(years_hug, mb_d, color=COL_DH,   lw=1.5, marker='s', ms=5, label='GloGEM Δh param.', zorder=3)
    ax.plot(years_hug, mb_f, color=COL_FLOW,  lw=1.5, marker='^', ms=5, label='GloGEMflow',       zorder=3)
    ax.set_title(name, fontsize=12, fontweight='bold')
    ax.set_xlabel('Year')
    ax.xaxis.set_major_locator(MaxNLocator(integer=True))
    if g == 0:
        ax.set_ylabel('Annual MB (m w.e. yr$^{-1}$)')
    ax.legend(loc='lower left', fontsize=9)

fig.tight_layout()
plt.show()

## 3. Geometry evolution

In [ ]:
fig, axes = plt.subplots(3, 2, figsize=(13, 11), sharex=True)

ylabels    = ['Area (km$^2$)', 'Volume (km$^3$)', 'ELA (m a.s.l.)']
data_pairs = [(area_dh, area_flow), (vol_dh, vol_flow), (ela_dh, ela_flow)]

for row, (ylabel, (d_dh, d_flow)) in enumerate(zip(ylabels, data_pairs)):
    for g, name in enumerate(GLACIER_NAMES):
        ax = axes[row, g]
        ax.plot(years, d_dh[g],   color=COL_DH,   lw=1.5, label='GloGEM Δh param.')
        ax.plot(years, d_flow[g], color=COL_FLOW,  lw=1.5, label='GloGEMflow')
        if row == 0:
            ax.set_title(name, fontsize=12, fontweight='bold')
        if g == 0:
            ax.set_ylabel(ylabel)
        if row == 2:
            ax.set_xlabel('Year')
            ax.xaxis.set_major_locator(MaxNLocator(integer=True))
        # Legend only once (top-right panel)
        if row == 0 and g == 1:
            ax.legend(loc='lower left', fontsize=9)

fig.tight_layout()
plt.show()

## 4. Mean surface elevation change rate (dh/dt) — Hugonnet period

Derived from model output as central-difference dh/dt = ΔV / (A · Δt).  
Hugonnet-implied dh/dt = MB / 0.9 (ice density ratio).

In [ ]:
def compute_dhdt(vol, area):
    """Central-difference dh/dt in m yr-1; vol in km3, area in km2."""
    dh = np.full_like(vol, np.nan)
    dh[:, 1:-1] = (vol[:, 2:] - vol[:, :-2]) * 1e9 / (2.0 * area[:, 1:-1] * 1e6)
    dh[:, 0], dh[:, -1] = dh[:, 1], dh[:, -2]
    return dh

dhdt_dh   = compute_dhdt(vol_dh,   area_dh)
dhdt_flow = compute_dhdt(vol_flow,  area_flow)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5), sharex=True, sharey=False)

for g, (ax, name) in enumerate(zip(axes, GLACIER_NAMES)):
    hug_dhdt     = hug_mb[g]  / 0.9
    hug_dhdt_err = hug_err[g] / 0.9
    ax.axhspan(hug_dhdt - hug_dhdt_err, hug_dhdt + hug_dhdt_err,
               color=COL_HUG, alpha=ALPHA_BAND, zorder=1, label='Hugonnet ±1σ')
    ax.axhline(hug_dhdt, color=COL_HUG, lw=1.5, ls='--', zorder=2, label='Hugonnet-implied')
    ax.plot(years_hug, dhdt_dh[g,   i_hug0:i_hug1], color=COL_DH,
            lw=1.5, marker='s', ms=5, label='GloGEM Δh param.', zorder=3)
    ax.plot(years_hug, dhdt_flow[g, i_hug0:i_hug1], color=COL_FLOW,
            lw=1.5, marker='^', ms=5, label='GloGEMflow',        zorder=3)
    ax.set_title(name, fontsize=12, fontweight='bold')
    ax.set_xlabel('Year')
    ax.xaxis.set_major_locator(MaxNLocator(integer=True))
    if g == 0:
        ax.set_ylabel('Mean dh/dt (m yr$^{-1}$)')
    ax.legend(loc='lower left', fontsize=9)

fig.tight_layout()
plt.show()

## 5. 2100 projections

In [ ]:
i_yr0  = 0
i_2100 = 2100 - TRAN0
i_2080 = 2080 - TRAN0

print(f"\n{'':=<80}")
print(f"{'2100 PROJECTION SUMMARY':^80}")
print(f"{'':=<80}")
v_col, a_col = 'V km$^3$', 'A km$^2$'
print(f"{'Glacier':<14} {'Model':<18} {v_col:>9} {'V%':>6} {a_col:>9} {'A%':>6} {'MB 2080-2100':>13}")
print('-' * 80)
for g, name in enumerate(GLACIER_NAMES):
    for label, vol, area, mb in [('GloGEM Δh param.', vol_dh,   area_dh,   mb_dh),
                                   ('GloGEMflow',       vol_flow,  area_flow, mb_flow)]:
        v0, v1 = vol[g, i_yr0], vol[g, i_2100]
        a0, a1 = area[g, i_yr0], area[g, i_2100]
        mb_late = mb[g, i_2080:i_2100+1].mean()
        print(f'{name:<14} {label:<18} {v1:>9.3f} {v1/v0*100:>6.1f} {a1:>9.2f} {a1/a0*100:>6.1f} {mb_late:>+13.3f}')
    print()

# Grouped bar chart
vfrac = np.array([[vol_dh[g,   i_2100] / vol_dh[g,   i_yr0],
                   vol_flow[g, i_2100] / vol_flow[g, i_yr0]] for g in range(2)]) * 100

fig, ax = plt.subplots(figsize=(7, 5))
x = np.array([0.0, 1.5])
w = 0.35
ax.bar(x - w/2, vfrac[:, 0], w, color=COL_DH,   label='GloGEM Δh param.')
ax.bar(x + w/2, vfrac[:, 1], w, color=COL_FLOW,  label='GloGEMflow')
ax.set_xticks(x)
ax.set_xticklabels(GLACIER_NAMES, fontsize=11)
ax.set_ylabel('Remaining volume (% of 1940 value)')
ax.set_title('Volume fraction remaining at 2100', fontsize=12)
ax.set_ylim(0, 105)
ax.legend(loc='upper right', fontsize=9)
fig.tight_layout()
plt.show()